# 💻 Notebook do Aluno — Aula 03: Structured Output e Pydantic v2

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 03/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🧱 BaseModel · PydanticOutputParser**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Fazer o LLM retornar dados num schema garantido pelo Pydantic — não em texto livre. Ao final, a chain do grupo aceita um input e retorna um objeto Python com campos validados e tipados.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama pydantic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Literal
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: defina a classe Pydantic para o domínio do grupo
# Mínimo 4 campos com tipos variados (str, int, List, Optional, Literal)
class NomeDaClasseDoGrupo(BaseModel):
    campo1: tipo = Field(description=___)
    campo2: tipo = Field(description=___)
    campo3: tipo = Field(description=___)
    campo4: Optional[tipo] = Field(None, description=___)

# 👉 LACUNA 2: crie o PydanticOutputParser com sua classe
parser = PydanticOutputParser(pydantic_object=___)

# 👉 LACUNA 3: crie o prompt com {format_instructions} e {pergunta}
# Use .partial() para pré-preencher format_instructions
prompt = ChatPromptTemplate.from_messages([
    ("system", ___),
    ("human",  ___),
]).partial(format_instructions=___)

llm   = ChatOllama(model="gpt-oss:120b", format="json")
# 👉 LACUNA 4: monte a chain e invoque com try/except
chain = ___ | ___ | ___

try:
    resultado = chain.invoke({"pergunta": ___})
    print(resultado.model_dump())
except ValidationError as e:
    print(f"Erro de schema: {e}")

---

## ✍️ Suas anotações

Registre aqui as observações da prática (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 03

Os quatro exercícios praticam schema Pydantic para o domínio do grupo — modelagem com Field e validator, correção de schema quebrado e comparação entre parsers. Complete os andaimes, rode cada célula no Colab e entregue o notebook com os schemas validados.


### Exercício 1 — Modele um schema Pydantic e valide na mão · ★★☆ · 10 min

*Individual · Colab*

Antes de plugar o LLM, prove que o schema valida: o mesmo `BaseModel` aceita o dado bom e reprova o inválido com erro detalhado.

1. Complete a classe — o tipo da lista de habilidades, o teto de `anos_exp` e os valores fixos do `Literal`.
2. Complete o validator: qual valor mínimo reprova `anos_exp`?
3. Rode o caminho bom (`model_dump()` com campos tipados) e o caminho ruim dentro do `try/except` — leia o campo e a mensagem em `e.errors()`.

> **💡 Dica:** no Pydantic v2, `Optional[tipo]` precisa de valor padrão — `Field(None, description=...)` — e o validator termina com `return v`.


In [ ]:
# 👉 LACUNA: modele o schema e prove a validação — caminho bom e caminho ruim
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import List, Optional, Literal

class Curriculo(BaseModel):
    nome:        str       = Field(description="Nome completo")
    email:       str       = Field(description="E-mail")
    habilidades: ___       = Field(description="Lista de habilidades")   # 👉 LACUNA 1: tipo de lista
    anos_exp:    int       = Field(ge=___, le=50, description="Anos de experiência")  # 👉 LACUNA 2: limite mínimo
    senioridade: Optional[Literal[___]] = Field(None, description="junior|pleno|senior")  # 👉 LACUNA 3: valores fixos

    @field_validator("anos_exp")
    @classmethod
    def anos_positivos(cls, v):
        if v < ___:                                  # 👉 LACUNA 4
            raise ValueError("anos_exp deve ser positivo")
        return v

# Caminho bom — instanciação válida, campos tipados
c = Curriculo(nome="Ana Silva", email="ana@email.com", habilidades=["Python", "LangChain"],
              anos_exp=3, senioridade="pleno")
print(type(c), "→", c.model_dump())

# Caminho ruim — valor fora do Literal e anos_exp como texto: erro campo por campo
try:
    Curriculo(nome="Carlos", email="c@email.com", habilidades=["SQL"],
              anos_exp="dez", senioridade="ninja")
except ValidationError as e:
    for erro in e.errors():
        print(erro["loc"], "→", erro["msg"])


### Exercício 2 — Schema quebrado: análise e correção no código · ★★☆ · 10 min

*Individual · Colab*

O schema abaixo está incompleto: tipo de lista ausente, `Literal` sem valores e validator com lacunas.

1. Rode a célula e leia o erro do Pydantic — qual campo reclama primeiro?
2. Complete os tipos e os `Field(description=...)` — descrições claras reduzem o erro do modelo.
3. Complete o validator, o parser e a chain com o `format_instructions` via `.partial(...)`, e confirme `type(r)` → sua classe.

> **💡 Dica:** no Pydantic v2, `Optional[tipo]` precisa de valor padrão — `Field(None, description=...)`.


In [ ]:
# 👉 LACUNA: o schema abaixo está incompleto — complete e rode
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import List, Optional, Literal

class Receita(BaseModel):
    nome:         str  = Field(description="Nome do prato")
    porcoes:      int  = Field(description="Rende quantas porções")
    ingredientes: ___  = Field(description="___")            # 👉 LACUNA: tipo de lista
    dificuldade:  Optional[Literal[___]] = Field(None, description="facil|media|dificil")  # 👉 LACUNA: valores fixos
    tempo_min:    int  = Field(description="Tempo total em minutos")

    # 👉 LACUNA: valide que tempo_min é positivo
    @field_validator("tempo_min")
    @classmethod
    def tempo_positivo(cls, v):
        if v <= ___:
            raise ValueError("tempo_min deve ser positivo")
        return ___

parser = PydanticOutputParser(pydantic_object=___)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um chef. Extraia a receita do texto.\n{format_instructions}"),
    ("human",  "{pergunta}"),
]).partial(format_instructions=___)          # 👉 LACUNA

chain = ___ | llm | parser                   # 👉 LACUNA: ordem correta da chain

try:
    r = chain.invoke({"pergunta": "Arroz com frango para 4 pessoas, rápido, ingredientes simples"})
    print(type(r))                            # esperado: <class 'Receita'>
    print(r.model_dump())
except ValidationError as e:
    print(f"Erros: {e.error_count()}")
    for erro in e.errors():
        print("  ", erro["loc"], "→", erro["msg"])


### Exercício 3 — Mesma pergunta, dois parsers · ★★☆ · 10 min

*Individual · Colab*

Duas chains idênticas, exceto pelo parser final — a diferença aparece no tipo do resultado e no tratamento de erro.

1. Complete os dois parsers nas chains — `JsonOutputParser` de um lado, `PydanticOutputParser` do outro.
2. Rode e compare `type(r1)` (dict) e `type(r2)` (objeto) — e o acesso por chave vs por atributo.
3. Envolva a chamada com `anos_exp="dez"` em `try/except ValidationError`: quem detecta o tipo errado — o parser JSON ou o Pydantic?

> **💡 Dica:** o `JsonOutputParser` para no JSON válido; o `PydanticOutputParser` só devolve quando o objeto passa na validação — a diferença aparece quando o modelo escorrega no schema.


In [ ]:
# 👉 LACUNA: mesmo prompt, dois parsers — compare os tipos
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser

pergunta = "Extraia: João Silva, joao@email.com, 5 anos de experiência, Python e LangChain"

# 👉 LACUNA 1: complete os parsers
chain_dict    = prompt | llm | ___   # parser que devolve dict puro
chain_pydantic = prompt | llm | ___  # parser que devolve objeto Pydantic

r1 = chain_dict.invoke({"curriculo": pergunta})
r2 = chain_pydantic.invoke({"curriculo": pergunta})

print(type(r1))   # esperado: <class 'dict'>      → acesso r1["nome"]
print(type(r2))   # esperado: <class 'Curriculo'> → acesso r2.nome
print(r1["nome"], "|", r2.nome)

# 👉 LACUNA 2: qual chain detecta anos_exp="dez"? Envolva a chamada da
# chain Pydantic em try/except ValidationError e imprima e.errors()


### Exercício 4 — Schema aninhado com validator · ★★☆ · 10 min

*Individual · Colab*

Modele um schema com objetos aninhados e validator para o domínio do grupo.

1. Complete os 3 valores fixos do `Literal` de categoria e os limites `ge/le` da nota.
2. Complete o validator — o intervalo válido e o `return v`.
3. Complete o parser e invoque a chain com `format="json"`: input bom sai tipado no `model_dump()`, input ruim dispara `ValidationError` com campo e mensagem.

> **💡 Dica:** o validator é `@field_validator("campo")` + `@classmethod` e precisa fazer `return v` no final.


In [ ]:
# 👉 LACUNA: schema aninhado com validator para o domínio do grupo
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import List, Literal

class Ponto(BaseModel):
    titulo:    str
    descricao: str

class Avaliacao(BaseModel):
    dominio:       str = Field(description="Domínio do grupo")
    pontos_fortes: List[Ponto]   # 👉 LACUNA: lista de objetos aninhados
    pontos_fracos: List[Ponto]
    categoria:     Literal[___]              # 👉 LACUNA: 3 valores fixos
    nota_final:    float = Field(___, description="Nota de 0 a 10")  # ge e le?

    # 👉 LACUNA: validator que reprova nota fora de 0..10
    @field_validator("nota_final")
    @classmethod
    def nota_valida(cls, v):
        if not (___ <= v <= ___):
            raise ValueError("nota fora do intervalo")
        return ___

parser = PydanticOutputParser(pydantic_object=___)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um avaliador. Responda no schema.\n{format_instructions}"),
    ("human",  "{pergunta}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | ChatOllama(model="gpt-oss:120b", format="json") | parser

try:
    r = chain.invoke({"pergunta": "Avalie: chatbot de ___ com memória e JSON"})
    print(r.model_dump())
except ValidationError as e:
    for erro in e.errors():
        print(erro["loc"], "→", erro["msg"])


## 📚 Referências da aula

- Docs Pydantic — BaseModel, Field, ValidationError (v2). docs.pydantic.dev/latest/concepts/models
- Docs LangChain — PydanticOutputParser: integração com Pydantic para saídas estruturadas. python.langchain.com/docs/modules/model_io/output_parsers/types/pydantic
- Docs LangChain — Structured output com with_structured_output() (abordagem alternativa moderna). python.langchain.com/docs/concepts/structured_outputs
- Docs Ollama — Structured outputs com format="json". ollama.com/blog/structured-outputs
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 22 — processamento de linguagem natural: a diferença entre extrair informação estruturada e gerar texto livre.

---

**→ Próxima Aula — Aula 04 · 24/08** — Context Engineering — de prompt engineering para context engineering
  
Gerenciar o contexto como recurso. CKP01 R4 + entrega.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*